# Industrial Equipment Failure Prediction (AI4I 2020 Dataset)

## Phase 1: Industrial Data Understanding (EDA)
**Objective**: Validate physical assumptions and understand failure behavior using the AI4I 2020 dataset.

### Dataset Description
- **Air temperature [K]**: Generated using a random walk process later normalized to a standard deviation of 2 K around 300 K
- **Process temperature [K]**: Generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K.
- **Rotational speed [rpm]**: Calculated from a power of 2860 W, overlaid with a normally distributed noise
- **Torque [Nm]**: Torque values are normally distributed around 40 Nm with a Ïƒ = 10 Nm and no negative values.
- **Tool wear [min]**: The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process.
- **Machine failure**: Label that indicates, whether the machine has failed in this particular datapoint for any of the following failure modes are true.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, recall_score, precision_score, roc_auc_score, f1_score, precision_recall_curve, auc, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import json
import shap

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

### 1.1 Data Loading & Sanity Checks

In [ ]:
df = pd.read_csv('ai4i2020.csv')

# Rename columns for easier access
df.columns = ['UDI', 'Product_ID', 'Type', 'Air_Temp', 'Process_Temp', 'Rot_Speed', 'Torque', 'Tool_Wear', 'Machine_Failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

print("Shape:", df.shape)
display(df.head())
print(df.info())

### 1.2 Target Analysis

In [ ]:
failure_rate = df['Machine_Failure'].mean() * 100
print(f"Global Failure Rate: {failure_rate:.2f}%")

# Failure Type Distribution
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
mode_counts = df[failure_modes].sum().sort_values(ascending=False)
print("\nFailure Mode Distribution:")
print(mode_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=mode_counts.index, y=mode_counts.values, palette='viridis')
plt.title('Distribution of Failure Modes')
plt.ylabel('Count')
plt.show()

### 1.3 Sensor Distributions & Relationships

In [ ]:
sensors = ['Air_Temp', 'Process_Temp', 'Rot_Speed', 'Torque', 'Tool_Wear']

plt.figure(figsize=(15, 10))
for i, col in enumerate(sensors):
    plt.subplot(2, 3, i+1)
    sns.kdeplot(data=df, x=col, hue='Machine_Failure', fill=True, common_norm=False)
    plt.title(f'{col} Distribution by Failure')
plt.tight_layout()
plt.show()

### 1.4 Machine Type Analysis

In [ ]:
type_failure = df.groupby('Type')['Machine_Failure'].mean() * 100
print("Failure Rate by Machine Type:")
print(type_failure)

plt.figure(figsize=(6, 4))
sns.barplot(x=type_failure.index, y=type_failure.values, palette='magma')
plt.ylabel('Failure Rate (%)')
plt.title('Failure Rate by Machine Quality Variant')
plt.show()

## Phase 2: Preprocessing & Feature Engineering
**Objective**: Convert raw data into physics-based failure signals.

**Engineered Features**:
1. **Temp_Diff**: Process Temp - Air Temp. (Indicator of cooling efficiency/heat dissipation)
2. **Power**: Torque * Rotational Speed * (2pi/60). (Mechanical power output)
3. **Wear_Rate**: Tool Wear / Rotational Speed. (Wear per revolution proxy)
4. **Torque_Temp_Ratio**: Torque / Process Temp.

In [ ]:
# Copy DF to avoid warnings
df_proc = df.copy()

# 1. Temp Diff
df_proc['Temp_Diff'] = df_proc['Process_Temp'] - df_proc['Air_Temp']

# 2. Power (P = 2*pi*N*T / 60) -> Watts
df_proc['Power'] = 2 * np.pi * df_proc['Rot_Speed'] * df_proc['Torque'] / 60

# 3. Wear Strain (Interaction feature)
df_proc['Wear_Strain'] = df_proc['Tool_Wear'] * df_proc['Torque']

# Check correlations with new features
new_features = ['Temp_Diff', 'Power', 'Wear_Strain']
plt.figure(figsize=(10, 8))
sns.heatmap(df_proc[sensors + new_features + ['Machine_Failure']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Including Engineered Features')
plt.show()

### 2.1 Splitting & Encoding

In [ ]:
# Drop ID columns and auxiliary targets (Failure Modes)
drop_cols = ['UDI', 'Product_ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X = df_proc.drop(['Machine_Failure'] + drop_cols, axis=1)
y = df_proc['Machine_Failure']

# One-Hot Encoding for Type
X = pd.get_dummies(X, columns=['Type'], drop_first=True)

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## Phase 3: Cost-Sensitive Modeling (Binary Failure)

In [ ]:
COST_FN = 1000
COST_FP = 100
COST_TP = 100
COST_TN = 0

def calculate_total_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (1, 1): return 0 
    tn, fp, fn, tp = cm.ravel()
    return (fn * COST_FN) + (fp * COST_FP) + (tp * COST_TP) + (tn * COST_TN)

scaler = StandardScaler()

pipeline = ImbPipeline([
    ('scaler', scaler),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(
        n_estimators=200, 
        learning_rate=0.05, 
        max_depth=6,
        eval_metric='logloss',
        random_state=42,
        use_label_encoder=False
    ))
])

pipeline.fit(X_train, y_train)

# Threshold Optimization
probs_val = pipeline.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0, 1, 100)
costs = []

for t in thresholds:
    preds = (probs_val >= t).astype(int)
    cost = calculate_total_cost(y_val, preds)
    costs.append(cost)

best_thresh = thresholds[np.argmin(costs)]
min_cost = min(costs)

print(f"Best Threshold: {best_thresh:.2f}")

# Evaluation on Test
probs_test = pipeline.predict_proba(X_test)[:, 1]
preds_test = (probs_test >= best_thresh).astype(int)
cm = confusion_matrix(y_test, preds_test)

print("--- BINARY MODEL RESULTS ---")
print(f"Recall: {recall_score(y_test, preds_test):.2f}")
print(f"ROC-AUC: {roc_auc_score(y_test, probs_test):.2f}")

# Save Binary Model
joblib.dump(pipeline, 'model_ai4i.joblib')
config = {'threshold': float(best_thresh), 'features': X.columns.tolist(), 'costs': {'FN': COST_FN, 'FP': COST_FP}}
with open('model_config_ai4i.json', 'w') as f: json.dump(config, f)

## Phase 5: Failure Mode Classification
Objective: Predict the specific failure mode (TWF, HDF, PWF, OSF, RNF) given that a failure occurred.

In [ ]:
# Prepare Multi-class Target
# Create a 'Failure_Type' column. If multiple, pick first valid (simplification) or use 'Machine failure' as 0.
def get_failure_type(row):
    if row['Machine_Failure'] == 0:
        return 'No Failure'
    for mode in ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']:
        if row[mode] == 1:
            return mode
    return 'Unknown'

df_proc['Failure_Class'] = df_proc.apply(get_failure_type, axis=1)

# Filter only samples that failed for training the Diagnostic Model
failed_df = df_proc[df_proc['Machine_Failure'] == 1].copy()
print("Failed Samples Distribution:")
print(failed_df['Failure_Class'].value_counts())

X_diag = failed_df.drop(['Machine_Failure', 'Failure_Class'] + drop_cols, axis=1)
X_diag = pd.get_dummies(X_diag, columns=['Type'], drop_first=True)
y_diag = failed_df['Failure_Class']

# Encode Target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_diag_encoded = le.fit_transform(y_diag)

# Train Diagnostic Model (Simple Random Forest due to low sample count)
from sklearn.ensemble import RandomForestClassifier
diag_model = RandomForestClassifier(n_estimators=100, random_state=42)
diag_model.fit(X_diag, y_diag_encoded)

# Evaluate
y_diag_pred = diag_model.predict(X_diag)
print("\nDiagnostic Model Report (Train Limit):")
print(classification_report(y_diag_encoded, y_diag_pred, target_names=le.classes_))

# Save Diagnostic Model
joblib.dump(diag_model, 'diag_model.joblib')
joblib.dump(le, 'diag_label_encoder.joblib')

## Phase 6: Interpretability (SHAP)
Understanding what drives the binary failure prediction.

In [ ]:
# SHAP expects the inner model of the pipeline
model_xgb = pipeline.named_steps['classifier']
X_test_scaled = pipeline.named_steps['scaler'].transform(X_test)
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)

explainer = shap.Explainer(model_xgb)
shap_values = explainer(X_test_df)

# Summary Plot
plt.figure()
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title("SHAP Feature Importance (Global)")
plt.tight_layout()
plt.show()

# Dependence Plot for Power
plt.figure()
shap.dependence_plot("Power", shap_values.values, X_test_df, show=False)
plt.show()